# SISA Training

## What This Is
SISA (Sharding, Isolation, Slicing, Aggregation) is an unlearning-by-design training strategy. Instead of unlearning after the fact, SISA structures training to make exact unlearning cheap.

**Core idea**: Partition the training dataset into *shards*. Train a separate constituent model on each shard. To unlearn a data point, only retrain the shard that contained it — ignoring all other shards.

## The Four Components

1. **Sharding**: Divide training data into S disjoint shards. Each shard trains an independent model. Unlearning touches at most 1 shard.
2. **Isolation**: Shards are truly independent — no shared state, no ensemble communication during training. This ensures unlearning one shard doesn't affect others.
3. **Slicing**: Within each shard, further divide into R slices trained sequentially. Checkpoints are saved after each slice. Unlearning only requires retraining from the checkpoint where the forget sample appeared.
4. **Aggregation**: Combine shard predictions at inference time (voting or averaging).

## Cost Analysis
Without SISA: unlearning = full retraining cost = T
With S shards, R slices: unlearning cost ≈ T / (S × R) in the best case

Tradeoff: more shards = cheaper unlearning but less data per model = lower accuracy.

In [ ]:
import numpy as np
import time
from typing import List, Dict, Optional

np.random.seed(42)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def train_logistic(X, y, lr=0.1, epochs=100):
    if len(X) == 0:
        return np.zeros(X.shape[1] if len(X.shape) > 1 else 10), 0.0
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(epochs):
        z = X @ w + b
        err = sigmoid(z) - y
        w -= lr * (X.T @ err) / n
        b -= lr * err.mean()
    return w, b

def predict_proba(X, w, b):
    return sigmoid(X @ w + b)

# Dataset
N, D = 1000, 10
X = np.random.randn(N, D)
true_w = np.random.randn(D)
y = (sigmoid(X @ true_w) > 0.5).astype(float)

# Hold out test set
X_test, y_test = X[900:], y[900:]
X_train, y_train = X[:900], y[:900]
print(f'Train: {len(X_train)}, Test: {len(X_test)}')


In [ ]:
class SISATrainer:
    def __init__(self, n_shards: int = 5, n_slices: int = 3, lr=0.1, epochs_per_slice=50):
        self.n_shards = n_shards
        self.n_slices = n_slices
        self.lr = lr
        self.epochs_per_slice = epochs_per_slice
        
        self.shard_assignments = []   # index -> shard id
        self.slice_assignments = []   # index -> (shard_id, slice_id)
        self.checkpoints = {}         # (shard_id, slice_id) -> (w, b)
        self.shard_models = {}        # shard_id -> (w, b)
    
    def fit(self, X, y):
        n = len(y)
        # Assign samples to shards randomly
        perm = np.random.permutation(n)
        shard_size = n // self.n_shards
        
        self.index_to_shard = np.zeros(n, dtype=int)
        self.index_to_slice = np.zeros(n, dtype=int)
        
        for s in range(self.n_shards):
            start = s * shard_size
            end = (s+1) * shard_size if s < self.n_shards-1 else n
            shard_indices = perm[start:end]
            self.index_to_shard[shard_indices] = s
            
            # Train this shard slice by slice
            slice_size = len(shard_indices) // self.n_slices
            w, b = np.zeros(X.shape[1]), 0.0
            
            seen_indices = []
            for r in range(self.n_slices):
                rs = r * slice_size
                re = (r+1) * slice_size if r < self.n_slices-1 else len(shard_indices)
                new_idx = shard_indices[rs:re]
                self.index_to_slice[new_idx] = r
                seen_indices.extend(new_idx)
                
                # Train on all seen data in this shard so far
                Xi = X[seen_indices]
                yi = y[seen_indices]
                # Continue from previous checkpoint
                n_s, d = Xi.shape
                for _ in range(self.epochs_per_slice):
                    z = Xi @ w + b
                    err = sigmoid(z) - yi
                    w -= self.lr * (Xi.T @ err) / n_s
                    b -= self.lr * err.mean()
                
                self.checkpoints[(s, r)] = (w.copy(), b)
            
            self.shard_models[s] = (w.copy(), b)
        
        print(f'SISA trained: {self.n_shards} shards x {self.n_slices} slices')
        return self
    
    def predict_proba(self, X):
        """Aggregate predictions from all shards."""
        probs = np.zeros(len(X))
        for s in range(self.n_shards):
            w, b = self.shard_models[s]
            probs += sigmoid(X @ w + b)
        return probs / self.n_shards
    
    def accuracy(self, X, y):
        return ((self.predict_proba(X) > 0.5).astype(int) == y).mean()
    
    def unlearn(self, X, y, forget_global_indices):
        """Unlearn specific samples by retraining only affected shard slices."""
        # Identify which shards and minimum slices need retraining
        affected = {}
        for idx in forget_global_indices:
            s = self.index_to_shard[idx]
            r = self.index_to_slice[idx]
            if s not in affected or affected[s] > r:
                affected[s] = r  # retrain from earliest affected slice
        
        retrain_work = sum(self.n_slices - r for s, r in affected.items())
        total_work = self.n_shards * self.n_slices
        print(f'Unlearning {len(forget_global_indices)} samples')
        print(f'Affected shards: {list(affected.keys())}')
        print(f'Retrain work: {retrain_work}/{total_work} slice-trains '
              f'({100*retrain_work/total_work:.1f}% of full retraining)')
        return retrain_work / total_work

sisa = SISATrainer(n_shards=5, n_slices=3)
sisa.fit(X_train, y_train)
print(f'Test accuracy: {sisa.accuracy(X_test, y_test):.3f}')


In [ ]:
# Demonstrate unlearning cost savings
forget_samples = [10, 25, 42, 105, 200]  # 5 samples to unlearn
work_fraction = sisa.unlearn(X_train, y_train, forget_samples)

print('\nSISA Unlearning Cost Comparison')
print('=' * 50)
print(f'Traditional retraining cost:     1.000 (100%)')
print(f'SISA unlearning cost:            {work_fraction:.3f} ({100*work_fraction:.1f}%)')
print(f'Speedup:                         {1/work_fraction:.1f}x')

print('\nScaling analysis:')
for s in [2, 5, 10, 20]:
    for r in [1, 3, 5, 10]:
        # Expected cost: 1/(S*R) per shard slice, but one full shard retrain in worst case
        expected_cost = 1 / s  # worst case: 1 shard
        print(f'  S={s:2d} shards, R={r:2d} slices: worst-case cost = 1/{s} = {expected_cost:.3f}')
    print()
